In [1]:
import duckdb
import polars as pl
import numpy as np
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


## Average session length (gap-based clustering)

In [2]:
year = 2025
df_gap = (
    df.sort(pl.col("date_played_unix"), descending=False)
    .filter(pl.col("track_played_utc").dt.year() == year)
    .with_columns(
        played_delta_s = pl.col("date_played_unix").diff().fill_null(0)
    )
).sort(pl.col("played_delta_s"), descending=True)
df_gap

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,played_delta_s
str,str,str,str,str,str,i64,datetime[μs],i64
"""kettama""","""e457531c-4add-45f0-9998-f65ec5…","""yosemite""","""""","""yosemite""","""""",1735894188,2025-01-03 08:49:48,131901
"""astrid sonne""","""bc6af694-5880-4770-bdf1-82d03f…","""great doubt""","""df87374d-53df-48d9-9677-ae3297…","""say you love me""","""b38a04eb-3543-4286-add0-a81315…",1754286170,2025-08-04 05:42:50,107218
"""fcukers""","""5e23e428-d9f5-4679-bd7c-8254d6…","""bon bon""","""7758de6f-d039-4672-ac4b-2c30fc…","""bon bon""","""864c5765-1c94-489a-80d7-7557de…",1752418527,2025-07-13 14:55:27,100913
"""fred again..""","""bca46a0c-25c9-42ca-98c2-e64c8a…","""actual life 2 (february 2 - oc…","""f50f4e97-8a6d-4212-a710-e82caf…","""roze (forgive)""","""9a3467f6-4fd8-45f8-a855-e552c0…",1766700505,2025-12-25 22:08:25,83148
"""yung lean""","""""","""forever yung""","""""","""forever yung""","""""",1740330680,2025-02-23 17:11:20,81702
…,…,…,…,…,…,…,…,…
"""caroline polachek""","""d8f43dc5-4613-48ac-8c23-a62b82…","""pang""","""3913f9b2-8622-4b0b-aeb5-3511d2…","""i give up""","""552cf74f-2de3-46c5-8adb-3955a0…",1765965150,2025-12-17 09:52:30,1
"""lover's skit""","""""","""all rights remixed""","""""","""no te metas - sassy 009 remix""","""""",1765965328,2025-12-17 09:55:28,1
"""amelie lens""","""67db280d-c3e4-49d9-978e-4050f4…","""higher ep""","""ab5756fb-afda-434e-906a-4fbca0…","""higher - fjaak remix""","""""",1766316371,2025-12-21 11:26:11,1


In [3]:
custom_bins = np.arange(0, 86400+1, 600)
bin_count, bins = np.histogram(
    df_gap.select(pl.col("played_delta_s")).to_numpy().flatten(),
    # 200
    custom_bins
)
print(bin_count)
print(bins)

[16843   791   389   256   216   175   148   100    89    76    63    54
    50    39    30    41    39    29    28    27    19    20    17    20
    13    11    15     9     8     7     7     2     3     2     4     2
     3     2     6     3     5     3     3     3     3     0     2     3
     2     8     7     4     2     3     4     7     7     3     7     9
     9    11    16     7    11     9    13    13    14    17     9     7
     5     8     8    14     7    10    10     6     5     1     6     8
     4     4    12     4     2     3     2     7     3     1     4     3
     2     1     2     2     2     1     2     2     1     1     0     0
     1     4     1     0     1     0     0     1     1     0     1     0
     1     0     0     0     0     0     1     0     0     0     0     0
     0     0     1     1     1     0     1     0     0     0     0     0]
[    0   600  1200  1800  2400  3000  3600  4200  4800  5400  6000  6600
  7200  7800  8400  9000  9600 10200 10800 11400 1

We will set 10 minutes (600s) as our threshold for what consitutes as a new listening session.

In [4]:
threshold = 20*60 # 3600
# threshold = 3600 # 3600
df_sessions = (
    df_gap
    .sort(pl.col("date_played_unix"), descending=False)
    .with_columns(
        (pl.col("played_delta_s") > threshold).alias("exceeds")
    )
    .with_columns(pl.col("exceeds").cum_sum().alias("nth_session"))
    .with_columns(
        pl.when(pl.col("exceeds"))
        .then(0)
        .otherwise(pl.col("played_delta_s"))
        .alias("played_delta_s")
    ) # set first song of session to 0
    .drop("exceeds")
    # count
    # .group_by(pl.col("track_played_utc"), pl.col("nth_session"))
    # .agg(
    #     n_scrobbles = pl.len()
    # )
    # .sort(pl.col("track_played_utc"), descending=False)
    # .with_columns(
    #     nth_scrobble = pl.col("n_scrobbles").cum_sum()
    # )
)

### Session statistics

In [5]:
df_session_stats = ( 
    df_sessions 
    .group_by(pl.col("nth_session"))
    .agg(
        # session_mean_s = pl.col("played_delta_s").mean(),
        tracks_played = pl.col("played_delta_s").count(),
        session_start = pl.col("track_played_utc").min(),
        session_end = pl.col("track_played_utc").max(),
    )
    .with_columns(
        session_length = (pl.col("session_end") - pl.col("session_start")),# .dt.total_seconds()
        date_played_utc = pl.col("session_end").dt.date()
    )
    # .filter(pl.col("session_length") > pl.duration(hours=3))
    .sort(pl.col("session_start"))
)
df_session_stats.sort(pl.col("session_length"), descending=True)

nth_session,tracks_played,session_start,session_end,session_length,date_played_utc
u32,u32,datetime[μs],datetime[μs],duration[μs],date
384,88,2025-02-27 07:24:08,2025-02-27 13:39:04,6h 14m 56s,2025-02-27
435,108,2025-03-07 07:10:54,2025-03-07 13:18:16,6h 7m 22s,2025-03-07
411,89,2025-03-03 06:50:11,2025-03-03 12:38:24,5h 48m 13s,2025-03-03
538,72,2025-03-21 08:13:38,2025-03-21 13:53:29,5h 39m 51s,2025-03-21
233,82,2025-02-03 08:17:12,2025-02-03 13:52:43,5h 35m 31s,2025-02-03
…,…,…,…,…,…
2400,1,2025-12-27 09:55:32,2025-12-27 09:55:32,0µs,2025-12-27
2401,1,2025-12-27 15:19:30,2025-12-27 15:19:30,0µs,2025-12-27
2408,1,2025-12-30 12:45:08,2025-12-30 12:45:08,0µs,2025-12-30


In [6]:
import plotly.express as px
fig = px.timeline(df_session_stats, x_start="session_start", x_end="session_end", color="session_length")
fig.show()

In [7]:
# Average session length
df_session_stats.select(pl.col("session_length").mean().alias("avg_session_length"))

avg_session_length
duration[μs]
27m 36s 512003µs


In [8]:
df_sessions.group_by(pl.col("nth_session")).agg(session_length_s = pl.col("played_delta_s").sum()/3600).sort("session_length_s", descending=True)

nth_session,session_length_s
u32,f64
384,6.248889
435,6.122778
411,5.803611
538,5.664167
233,5.591944
…,…
2209,0.0
1956,0.0
1807,0.0


In [9]:
( 
    df_sessions 
    .group_by(pl.col("track_played_utc").dt.date().alias("date_played_utc"))
    .agg(
        n_sessions = pl.col("nth_session").unique().len()
    )
    # .sort(pl.col("track_played_utc"), descending=False)
    .sort(pl.col("n_sessions"), descending=True)
).join(df_session_stats, "date_played_utc", how="right").sort("n_sessions", descending=False).sort("session_length", descending=True)

n_sessions,nth_session,tracks_played,session_start,session_end,session_length,date_played_utc
u32,u32,u32,datetime[μs],datetime[μs],duration[μs],date
5,384,88,2025-02-27 07:24:08,2025-02-27 13:39:04,6h 14m 56s,2025-02-27
9,435,108,2025-03-07 07:10:54,2025-03-07 13:18:16,6h 7m 22s,2025-03-07
4,411,89,2025-03-03 06:50:11,2025-03-03 12:38:24,5h 48m 13s,2025-03-03
7,538,72,2025-03-21 08:13:38,2025-03-21 13:53:29,5h 39m 51s,2025-03-21
8,233,82,2025-02-03 08:17:12,2025-02-03 13:52:43,5h 35m 31s,2025-02-03
…,…,…,…,…,…,…
13,481,1,2025-03-14 05:52:46,2025-03-14 05:52:46,0µs,2025-03-14
13,485,1,2025-03-14 10:59:49,2025-03-14 10:59:49,0µs,2025-03-14
13,486,1,2025-03-14 11:32:31,2025-03-14 11:32:31,0µs,2025-03-14


In [10]:

df_sessions.filter(pl.col("track_played_utc").dt.date() == pendulum.datetime(2025, 1, 20).naive())

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,played_delta_s,nth_session
str,str,str,str,str,str,i64,datetime[μs],i64,u32
"""dj suzy""","""""","""digital girl""","""""","""digital girl - funk remix""","""""",1737365208,2025-01-20 09:26:48,0,120
"""mk.gee""","""262cf0ce-2af2-4686-a8cc-d6b33e…","""two star & the dream police""","""ec873d45-a37c-4738-b817-36d8cc…","""are you looking up""","""6c1cc859-c978-4db2-be60-46ce72…",1737365382,2025-01-20 09:29:42,174,120
"""bassvictim""","""""","""curse is lifted (club rmx)""","""""","""curse is lifted - club rmx""","""""",1737365548,2025-01-20 09:32:28,166,120
"""charli xcx""","""260b6184-8828-48eb-945c-bc4cb6…","""pop 2""","""5414958b-12e2-4a6e-b335-26002c…","""porsche (feat. mø)""","""fd6f8827-ee1c-46cf-b323-45e460…",1737365768,2025-01-20 09:36:08,220,120
"""snow strippers""","""33243fef-ce1a-4ba7-a81e-ec72e4…","""april mixtape 2""","""3c39c467-3e37-4a41-b530-b68429…","""you could be the one""","""7eff0928-a87b-436e-bff7-dd7e7f…",1737365976,2025-01-20 09:39:36,208,120
…,…,…,…,…,…,…,…,…,…
"""dj tired""","""""","""rest""","""""","""so what?""","""""",1737398141,2025-01-20 18:35:41,159,125
"""travis scott""","""e4a51f17-a57b-47b1-b37b-f552d0…","""astroworld""","""""","""stargazing""","""d96bd8fa-d0e7-4c5e-9a70-753778…",1737398542,2025-01-20 18:42:22,401,125
"""rico nasty""","""f7d8f1ce-6a72-422a-b746-22137b…","""iphone""","""4c531ee0-9f01-40d0-b608-0683ba…","""iphone""","""0921518d-5ef0-4450-8dfb-a85fdd…",1737398813,2025-01-20 18:46:53,271,125


In [11]:
month = 2
(
    df
    .filter(
        pl.col("track_played_utc").dt.year() == 2025,
        pl.col("track_played_utc").dt.month() == month
    )
    .group_by(
        pl.col("track_played_utc").dt.date(),
        pl.col("track_name"),
        pl.col("artist_name")
    )
    .agg(n_scrobbles = pl.len())
    .with_columns(daily_scrobbles=pl.col("n_scrobbles").sum().over("track_played_utc"))
    # .sort("n_scrobbles", descending=True)
    .sort([ "n_scrobbles", "daily_scrobbles" ], descending=[ True, True ])
    # .row(0, named=True)
)

track_played_utc,track_name,artist_name,n_scrobbles,daily_scrobbles
date,str,str,u32,u32
2025-02-07,"""get used to it (losin' it)""","""fromtheheart""",10,164
2025-02-25,null,null,9,44
2025-02-03,"""waited all night""","""jamie xx""",6,110
2025-02-20,"""hide away""","""riot shift""",6,104
2025-02-24,"""with you""","""bassvictim""",6,102
…,…,…,…,…
2025-02-05,"""bizarre love triangle - 2024 d…","""new order""",1,9
2025-02-05,"""life""","""jamie xx""",1,9
2025-02-05,"""hum it to google""","""bassvictim""",1,9


In [12]:
lastfm_df = pl.read_parquet(Path("../data/lastfm/lastfm-listening-2021-2026march.parquet"))
spotify_df = pl.read_parquet(Path("../data/spotify/2017-2021 spotify data.parquet"))
lastfm_df = lastfm_df.with_columns(spotify_track_uri=pl.lit(""))
spotify_df = spotify_df.with_columns(
        artist_mbid=pl.lit(""), album_mbid=pl.lit(""), track_mbid=pl.lit("")
    )
cols = [
        "track_played_utc",
        "date_played_unix",
        "track_name",
        "artist_name",
        "album_name",
        "track_mbid",
        "artist_mbid",
        "album_mbid",
        "spotify_track_uri",
    ]
lastfm_df = lastfm_df.select(cols)
spotify_df = spotify_df.select(cols)

df = lastfm_df.vstack(spotify_df).sort(
    pl.col("date_played_unix"), descending=False
)

In [ ]:
records = (
    df
    .filter(
        pl.col("track_played_utc").dt.year() >= 2021,
        pl.col("track_name").drop_nulls()
    )
    .unique(subset=["track_name"], keep="first")
    .sort("track_played_utc", descending=False)
).to_dicts()

In [15]:
(
    df
    .filter(
        pl.col("track_played_utc").dt.year() >= 2021,
    )
    .unique(subset=["track_name"], keep="first")
    .sort("track_played_utc", descending=False)
    .drop_nulls(pl.col("track_name"))
)

track_played_utc,date_played_unix,track_name,artist_name,album_name,track_mbid,artist_mbid,album_mbid,spotify_track_uri
datetime[μs],i64,str,str,str,str,str,str,str
2021-01-01 00:15:25,1609460125,"""NOT FAIR (feat. Corbin)""","""The Kid LAROI""","""F*CK LOVE""","""""","""""","""""","""spotify:track:2HrMDMckfCyM4OcZ…"
2021-01-01 00:18:55,1609460335,"""Her and Her Friend""","""TV Girl""","""French Exit""","""""","""""","""""","""spotify:track:4MwYkW85fXhNvV2w…"
2021-01-01 12:40:35,1609504835,"""Dorothy""","""Her's""","""Songs of Her's""","""""","""""","""""","""spotify:track:7jay75cMfpEIyIkk…"
2021-01-01 12:44:10,1609505050,"""Punk Monk""","""Playboi Carti""","""Whole Lotta Red""","""""","""""","""""","""spotify:track:5fSZXFOZfPVW9gvn…"
2021-01-01 12:47:26,1609505246,"""I hope that u think of me""","""Pity Party (Girls Club)""","""I hope that you think of me""","""""","""""","""""","""spotify:track:6z2EwIB2wTLdXJZ6…"
…,…,…,…,…,…,…,…,…
2026-03-23 17:54:33,1774288473,"""(Something) - 1""","""The Microphones""","""The Glow, Pt. 2""","""17261347-f3fa-40a8-81e7-4db50d…","""6f2f87e7-3ce9-4f1c-81dd-6f068a…","""3d341648-5aee-4128-805a-a9a0ef…",""""""
2026-03-23 18:09:24,1774289364,"""I'll Not Contain You""","""The Microphones""","""The Glow, Pt. 2""","""20abad51-57c0-4a9d-832d-0418fd…","""6f2f87e7-3ce9-4f1c-81dd-6f068a…","""3d341648-5aee-4128-805a-a9a0ef…",""""""
2026-03-23 18:12:51,1774289571,"""Darkness""","""Chris Stussy""","""Darkness""","""""","""5b7f4511-1764-4d80-83b5-8c150f…","""56464592-69b9-4ceb-8b58-705e18…",""""""
